In [3]:
import pandas as pd

In [ ]:
from google.colab import files
uploaded = files.upload()

In [8]:
movies = pd.read_csv('movies_cleaned.csv')
genres = pd.read_csv('genres_cleaned.csv')
movie_genres = pd.read_csv('movie_genres_cleaned.csv')
cast = pd.read_csv('cast_cleaned.csv')
crew = pd.read_csv('crew_cleaned.csv')
movie_keywords = pd.read_csv('movie_keywords_cleaned.csv')

In [9]:
print(movies.shape)
print(genres.shape)
print(movie_genres.shape)
print(cast.shape)
print(crew.shape)
print(movie_keywords.shape)

(2503, 12)
(19, 2)
(6878, 2)
(24902, 6)
(7289, 6)
(41681, 3)


In [10]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2503 entries, 0 to 2502
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   movie_id           2503 non-null   int64  
 1   title              2503 non-null   object 
 2   original_language  2503 non-null   object 
 3   release_date       2503 non-null   object 
 4   runtime            2503 non-null   int64  
 5   budget             2503 non-null   float64
 6   revenue            2503 non-null   float64
 7   popularity         2503 non-null   float64
 8   vote_average       2503 non-null   float64
 9   vote_count         2503 non-null   int64  
 10  status             2503 non-null   object 
 11  overview           2503 non-null   object 
dtypes: float64(4), int64(3), object(5)
memory usage: 234.8+ KB


In [14]:
movies['release_date'] = pd.to_datetime(movies['release_date'], format = '%Y-%m-%d')

In [15]:
movies['release_date'].dtype

dtype('<M8[ns]')

In [19]:
#1.List the total number of unique movies, genres, cast members and crew members collected across all tables.

print("Total Unique Movies:", movies['movie_id'].nunique())
print("Total Unique Genres:", genres['genre_id'].nunique())
print("Total Unique Cast Members:", cast['person_id'].nunique())
print("Total Unique Crew Members:", crew['person_id'].nunique())

Total Unique Movies: 2503
Total Unique Genres: 19
Total Unique Cast Members: 11300
Total Unique Crew Members: 2848


In [ ]:
# 2.List all movies with missing or zero budget/revenue values, and decide how these should be treated for the rest of the analysis.

missing_movies = movies[
    (movies['budget'].isna()) |
    (movies['budget'] == 0) |
    (movies['revenue'].isna()) |
    (movies['revenue'] == 0)
]

missing_movies[['movie_id', 'title', 'budget', 'revenue']]

In [ ]:
# 3.List the profit (revenue − budget) and return on investment for every movie that has valid budget and revenue figures.

all_movies = movies[(movies['budget'] > 0) & (movies['revenue'] > 0)]
all_movies['profit'] = all_movies['revenue'] - all_movies['budget']
all_movies['ROI'] = (all_movies['profit'] / all_movies['budget']) * 100

all_movies[['title', 'budget', 'revenue', 'profit', 'ROI']]

In [ ]:
# 4.List the top 15 movies by return on investment considering only movies with a budget of at least $1 million.

movies['ROI'] = ((movies['revenue'] - movies['budget']) / movies['budget']) * 100

top_movies = movies[movies['budget'] >= 1000000]
top_movies = top_movies.sort_values('ROI', ascending=False)

top_movies[['title', 'budget', 'revenue', 'ROI']].head(15)

In [ ]:
# 5.List the number of movies released per year and identify which year had the highest movie output in the dataset.

movies['year'] = movies['release_date'].dt.year

yearly_movies = movies['year'].value_counts().reset_index()

yearly_movies.columns = ['Year', 'Number of Movies']

highest_year = yearly_movies.loc[yearly_movies['Number of Movies'].idxmax()]

pd.DataFrame([highest_year])

In [ ]:
# 6.List all movies whose runtime is a statistical outlier compared to the rest of the dataset.

q1 = movies['runtime'].quantile(0.25)
q3 = movies['runtime'].quantile(0.75)
iqr= q3 - q1

print('iqr')

upper_limit = q3 + 1.5 * iqr
lower_limit = q1 - 1.5 * iqr

print(upper_limit)
print(lower_limit)

outliers = movies[(movies['runtime'] < lower_limit) |(movies['runtime'] > upper_limit)]
outliers[['title', 'runtime']]


In [ ]:
# 7.List each movies primary genre (the first genre listed).

primary_genre['genre_name'] = primary_genre['genre_id'].map(genres.set_index('genre_id')['genre_name'])
primary_genre['title'] = primary_genre['movie_id'].map(movies.set_index('movie_id')['title'])

primary_genre[['movie_id', 'title', 'genre_name']]

In [ ]:
# Compare average popularity across primary genres.

primary_genre['popularity'] = primary_genre['movie_id'].map(movies.set_index('movie_id')['popularity'])
average_popularity = primary_genre.groupby('genre_name')['popularity'].mean().reset_index()
average_popularity = average_popularity.sort_values(by='popularity', ascending=False)

display(average_popularity)

In [ ]:
# 8.List the top 10 most profile actors by number of movies, along with their average movie rating across those movies.

actor_movies = cast['person_id'].value_counts().head(10).reset_index()
actor_movies.columns = ['person_id', 'Number of Movies']
actor_movies['actor_name'] = actor_movies['person_id'].map(cast.drop_duplicates('person_id').set_index('person_id')['actor_name']
)
for i in range(len(actor_movies)):

    person_id = actor_movies.loc[i, 'person_id']
    movie_ids = cast[cast['person_id'] == person_id]['movie_id']
    ratings = movies[movies['movie_id'].isin(movie_ids)]['vote_average']
    actor_movies.loc[i, 'Average Rating'] = ratings.mean()

actor_movies[['actor_name', 'Number of Movies', 'Average Rating']]


In [ ]:
# 9.List any movie titles that appear more than once in the dataset, along with their release years, to check for duplicates or remakes.

remakes = movies[movies.duplicated('title', keep=False)]
remakes.loc[:, 'release_year'] = remakes['release_date'].dt.year
remakes = duplicate_titles.sort_values('title')

remakes[['title', 'release_year']]

In [ ]:
# 10.List movies whose popularity score and vote count

high_popularity = movies['popularity'].quantile(0.90)
low_popularity = movies['popularity'].quantile(0.10)

high_votes = movies['vote_count'].quantile(0.90)
low_votes = movies['vote_count'].quantile(0.10)

result = movies[
    ((movies['popularity'] > high_popularity) & (movies['vote_count'] < low_votes)) | #High popularity but low vote count
    ((movies['popularity'] < low_popularity) & (movies['vote_count'] > high_votes))   #Low popularity but high vote count
]
result[['title', 'popularity', 'vote_count']]